In [ ]:
from pathlib import Path
import os
from os import listdir as ls
import arviz as az
import pickle
import pycountry
import pycountry_convert as pc
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import h5py  # Make sure h5py initialises properly before pandas ruins it
from IPython.display import display, Markdown, Latex

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_prior_multipost
from emu_renewal.utils import get_countries_by_continent, get_job_commits_df

set_matplotlib_formats("svg")

# Purpose
This document presents comparisons of parameter prior distributions 
to their corresponding posterior distributions.

In [ ]:
display(Latex(r"\newpage"))

# Prior posterior comparisons by continent and country

In [ ]:
#| fig-align: center
plt.style.use("ggplot")
job_path = OUTPUTS_PATH / "48930936"
# all_countries = ls(job_path)
all_countries = [iso3 for iso3 in ls(job_path) if "oxcgrt" in ls(job_path / iso3) and "no_mob" in ls(job_path / iso3)]
countries_by_cont = get_countries_by_continent(all_countries)
for cont, countries in countries_by_cont.items():
    display(Latex(r"\newpage"))
    display(Markdown(f"\n## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c, country in enumerate(countries):
        country_name = pycountry.countries.lookup(country).name
        if c:
            display(Latex(r"\newpage"))
        display(Markdown(f"### {country_name}"))
        analyses = {d.name: Path(d.path) for d in os.scandir(job_path / country) if d.is_dir()}
        no_mob_path = analyses["no_mob"]
        targs = load_targets(no_mob_path)
            
        priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
        idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
        var_names = ["start"] + [v[5:] for v in targs.keys() if v.startswith("prop_")]
        display(plot_prior_multipost(var_names, priors, idatas, 4))

{{< pagebreak >}}

# Commits used for analyses
For reproducbility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_job_commits_df(job_path, all_countries).to_markdown())